# DLLM English <-> Hindi — Model Testing

Requires inputs:
- **dllm-code** dataset (the repo zip)
- **dllm-data** dataset (optional, for BLEU eval on the test TSVs)
- **checkpoint** (one of):
  - the committed *output* of the training notebook (Add Input -> Kaggle -> your committed notebook), or
  - `best_model.pt` / `resume.pt` uploaded as a dataset

The checkpoint is auto-located below — just run all cells.

In [ ]:
!pip install -q --upgrade transformers datasets python-Levenshtein pyyaml tqdm sacrebleu

In [ ]:
import os, glob, shutil, sys

# Kaggle auto-extracts uploaded zips, so inputs are directories of files.
# Symlink their contents into /kaggle/working (fall back to copy if needed).
os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

for ds in ['dllm-code', 'dllm-data']:
    src = f'/kaggle/input/{ds}'
    if not os.path.isdir(src):
        print(f'{ds}: not attached, skipping')
        continue
    print(f'{ds}: {os.listdir(src)}')
    for item in os.listdir(src):
        dst = os.path.join('/kaggle/working', item)
        if os.path.exists(dst):
            print(f'  {item}: already present, skipping')
            continue
        try:
            os.symlink(os.path.join(src, item), dst)
            print(f'  linked {item}')
        except OSError:
            s = os.path.join(src, item)
            if os.path.isdir(s):
                shutil.copytree(s, dst)
            else:
                shutil.copy2(s, dst)
            print(f'  copied {item}')

sys.path.insert(0, '/kaggle/working')
print(os.listdir('/kaggle/working'))

In [ ]:
# Auto-locate the best checkpoint. Multiple attached inputs may each contain a
# best_model.pt (e.g. an old tiny_shakespeare notebook output) — print ALL
# candidates with their trained step and pick the newest one.
import glob, torch, os

cands = sorted(glob.glob('/kaggle/input/**/best_model.pt', recursive=True) +
               glob.glob('/kaggle/working/**/best_model.pt', recursive=True))
for c in cands:
    try:
        step = torch.load(c, map_location='cpu').get('global_step', '?')
    except Exception as e:
        step = f'error: {e}'
    print(f'{c}  -> step {step}')

if cands:
    CKPT = max(cands, key=os.path.getmtime)  # newest file wins
else:
    CKPT = None
print('\nusing:', CKPT)
assert CKPT is not None, 'No best_model.pt found — add the training notebook output or a dataset'

In [ ]:
# Load the model once; `translate()` is then usable from any cell
import torch, yaml
from scripts.translate import load_model

config = yaml.safe_load(open('configs/translation_kaggle2.yaml'))
tok, model, inference = load_model(config, CKPT, 'cuda' if torch.cuda.is_available() else 'cpu')

def translate(text: str) -> str:
    return inference.generate(text, max_iterations=20).strip()

In [ ]:
# Smoke test — both directions
for s in ['How are you?', 'Where is the train station?', 'The weather is beautiful today.']:
    print('EN :', s)
    print('HI :', translate(s))
    print()
for s in ['तुम कैसे हो?', 'रेलवे स्टेशन कहाँ है?', 'मैं कल दिल्ली जा रहा हूँ।']:
    print('HI :', s)
    print('EN :', translate(s))
    print()

## BLEU evaluation (held-out OPUS-100 test sets)
`--n` controls the number of pairs scored.

In [ ]:
import sacrebleu

N = 200
for path, name in [('data/en_hi_test.tsv', 'en -> hi'), ('data/en_hi_rev_test.tsv', 'hi -> en')]:
    pairs = [l.rstrip('\n').split('\t') for l in open(path, encoding='utf-8') if '\t' in l][:N]
    hyps, refs = [], []
    for src, ref in pairs:
        hyps.append(translate(src))
        refs.append(ref)
    score = sacrebleu.corpus_bleu(hyps, [refs])
    print(f'{name}: BLEU = {score.score:.2f}')
    for i in range(min(3, N)):
        print(f'  SRC: {pairs[i][0]}\n  REF: {refs[i]}\n  OUT: {hyps[i]}')

## Debugging: refinement trajectory
Shows the canvas evolving over iterations (tag counts + text per step) — useful when output is wrong.

In [ ]:
res = inference.generate('How are you?', max_iterations=20, return_trajectory=True)
for step in res['trajectory']:
    c = step['tag_counts']
    print(f"step {step['step']:02d} K={c['KEEP']:3d} REP={c['REPLACE']:3d} DEL={c['DELETE']:3d} INS={c['INSERT']:3d} EXP={c['EXPAND']:3d} | {step['response_only'][:60]}")
print('\nfinal:', res['response_only'])

## Interactive
Use `translate('...')` in any new cell below.